In [0]:
%sql
--Grant Privileges
GRANT READ FILES, WRITE FILES,
      CREATE EXTERNAL TABLE, CREATE EXTERNAL VOLUME
  ON EXTERNAL LOCATION intellibi_saleslake_ext
  TO `account users`;
 
GRANT BROWSE
  ON EXTERNAL LOCATION intellibi_saleslake_ext
  TO `account users`;
 
SHOW GRANTS ON EXTERNAL LOCATION intellibi_saleslake_ext;


In [0]:
# List the bucket
display(dbutils.fs.ls("s3://saleslake-610639371366-us-east-2/saleslake/"))

In [0]:
# Create a folder + write a file
S3 = "s3://saleslake-610639371366-us-east-2/saleslake/dev/source_files/smoke/"
dbutils.fs.mkdirs(S3)
dbutils.fs.put(S3 + "hello.txt", "hello from databricks", overwrite=True)
print(dbutils.fs.head(S3 + "hello.txt"))


In [0]:
# Clean up
dbutils.fs.rm("s3://saleslake-610639371366-us-east-2/saleslake/dev/source_files/smoke/", recurse=True)
print("Smoke test passed end-to-end.")


In [0]:
%sql
CREATE TABLE IF NOT EXISTS saleslake_dev.bronze_dev.rawInvoice (
  invoice_id       STRING,
  invoice_number   STRING,
  customer_id      STRING,
  invoice_date     STRING,
  due_date         STRING,
  subtotal_amount  STRING,
  discount_code    STRING,
  discount_amount  STRING,
  tax_amount       STRING,
  total_amount     STRING,
  payment_status   STRING,
  payment_method   STRING,
  payment_date     STRING,
  currency         STRING,
  region           STRING,
  store_id         STRING,
  channel          STRING,
  created_by       STRING,
  source_file_name STRING,
  ingest_ts        TIMESTAMP
)
USING DELTA
LOCATION 's3://saleslake-610639371366-us-east-2/saleslake/dev/bronze_delta_tables/invoice/';


In [0]:
%sql
DESCRIBE EXTENDED saleslake_dev.bronze_dev.rawInvoice;


In [0]:
%sql
COPY INTO saleslake_dev.bronze_dev.rawInvoice
FROM (
  SELECT
    invoice_id, invoice_number, customer_id,
    invoice_date, due_date,
    subtotal_amount, discount_code, discount_amount, tax_amount, total_amount,
    payment_status, payment_method, payment_date,
    currency, region, store_id, channel, created_by,
    _metadata.file_name AS source_file_name,
    current_timestamp() AS ingest_ts
  FROM 's3://saleslake-610639371366-us-east-2/saleslake/dev/source_files/invoice/'
)
FILEFORMAT     = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false')
COPY_OPTIONS   ('mergeSchema' = 'false');



In [0]:
display(dbutils.fs.ls("s3://saleslake-610639371366-us-east-2/saleslake/dev/source_files/invoice/"))